Bei anderen NB doppelte resample ünberprüfen dank prune

## Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import model_analysis

import importlib
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd

import numpy as np

import cartopy.crs as ccrs
from scipy.stats import beta



## Load Data

In [2]:
max_radius = 4
hist = 2
is_local_data = True
Month_idx = 4
safe = True
start_wanted = None  # later this will be shifted, if it is to close to the beginning of the data, such that there is allways data also for the hist dimension
end = None
max_depth = 3


In [3]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [4]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")


In [5]:

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
approx_maximas = max_v


In [6]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

 

In [7]:
#min_start 

data_start = pd.to_datetime(raw_mrsol_for_mean.time[0].item())
offset = pd.tseries.frequencies.to_offset("ME")
min_start = data_start + pd.DateOffset(months=hist)

if start_wanted is None:
    start = min_start
else:
    start = pd.to_datetime(start_wanted) 
    start = max(min_start,start)
start = start.strftime("%Y-%m-%d")  

some explantions
slice 0-4 weil im moment die letzte schicht hartnäckig probleme macht

In [8]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = (ds/maximas).clip(max = 1-1e-15)
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [9]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [10]:
# Transform already as one function
#model.transform.Logit_Transform_ds()

In [11]:
def shape_input(ds, chunk_mask, hist, radius):
    ds = model.shape_data.add_hist_dimension(ds, hist)    
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.shape_data.add_radius_dimension(ds, radius)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

In [12]:
#The chunk_mask schould all be the same, this is important that there are no shape problems in the regressions, you can test this with  
#(chunk_mask != chunk_mask_2).sum(),#(chunk_mask != chunk_mask_1).sum(),
#The chunk_mask is used, that the regression can predict a depth.size vector at each gridcell, the masks: detail_mask_mean, detail_mask_var and detail_mask_test are later used to determan which points are real predictions and witch where just placeholders to let the regression run smoothly.

### Linear Regression of the mean

In [13]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [14]:
mean_predictor_sets  = {}
mean_max_predictor = shape_input(raw_input_for_mean,chunk_mask,hist, max_radius)
mean_max_predictor
for radius in range(0, max_radius):
   mean_predictor_sets[f"radius_{radius}"] = mean_max_predictor.sel(lon_translations=slice(-radius,radius)).sel(lat_translations=slice(-radius,radius))

In [15]:
mean_predictor_sets["radius_2"]

<xarray.Dataset> Size: 50MB
Dimensions:           (hist: 2, time: 165, lon_translations: 5,
                       lat_translations: 5, gridcell: 378)
Coordinates:
  * hist              (hist) int64 16B 0 1
  * time              (time) datetime64[ns] 1kB 1850-04-30 ... 2014-04-30
  * lon_translations  (lon_translations) int64 40B -2 -1 0 1 2
  * lat_translations  (lat_translations) int64 40B -2 -1 0 1 2
    lat               (gridcell) float64 3kB -51.75 -47.25 ... 78.75 78.75
    lon               (gridcell) float64 3kB 288.0 288.0 288.0 ... 279.0 297.0
    height            float64 8B 2.0
Dimensions without coordinates: gridcell
Data variables:
    tas               (lat_translations, lon_translations, time, hist, gridcell) float64 25MB ...
    pr                (lat_translations, lon_translations, time, hist, gridcell) float64 25MB ...
Attributes: (12/52)
    CDI:                       Climate Data Interface version 1.9.6 (http://m...
    history:                   Wed Jul 15 10:13:55 2026: cdo remapbil,r40x40 ...
    source:                    MPI-ESM1.2-LR (2017): \naerosol: none, prescri...
    institution:               Max Planck Institute for Meteorology
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    ...                        ...
    cmor_version:              3.5.0
    tracking_id:               hdl:21.14100/6b679cba-17b8-45eb-90dc-23d170c1998c
    cmip6-ng:                  \ncontact = cmip6-archive@env.ethz.ch\ndescrip...
    original_file_names:       /net/atmos/data/cmip6/historical/Amon/tas/MPI-...
    original_file_hash_codes:  44b9ee9e68daceb1f50e9680dcfb7744f0e272d73c5e18...
    CDO:                       Climate Data Operators version 1.9.6 (http://m...

In [16]:
Regr_set_mean = {}

In [17]:
#mean_target_da

In [18]:
for key,predictors in mean_predictor_sets.items():
    Regr_set_mean[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist","lon_translations","lat_translations"])
    Regr_set_mean[key].fit(predictors=predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

In [19]:
#Regr_set_mean["kontrolle"] = model.stats._parallel_linear_regression.ParLinearRegression()
#Regr_set_mean["kontrolle"].fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

Hier wäre die meinung einen neuen Run zu verwenden, die frage ist ob sich die variance durch das fehlen des runs zu tief ausfällt. Overfitting korrektur mit 1/(1-param/n_samples)^2? (SPäter probieren)

In [20]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)#Hier können noch sehr grosse werte auftauchen, wenn in irgendwelchen schichten die Maximas der verschieden runs sehr unterschiedlich sind.

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [21]:
var_predictor_sets  = {}
var_max_predictor = shape_input(raw_input_for_var,chunk_mask,hist,max_radius)

for radius in range(0, max_radius):
    var_predictor_sets[f"radius_{radius}"] = var_max_predictor.sel(lon_translations=slice(-radius,radius)).sel(lat_translations=slice(-radius,radius))

In [22]:
Regr_set_mean["radius_1"].params.pr

<xarray.DataArray 'pr' (gridcell: 378, depth: 3, hist: 2, lon_translations: 3,
                        lat_translations: 3)> Size: 163kB
array([[[[[ 2.97044159e+02,  2.08739363e+03,  1.68964413e+03],
          [-9.44307499e+02,  1.07368793e+04, -5.26417253e+03],
          [ 2.21124123e+03, -1.54037423e+03,  5.66895715e+02]],

         [[ 1.32186507e+03,  2.28888791e+02, -7.61839682e+02],
          [ 8.59854350e+02,  1.25770603e+03,  3.14035821e+02],
          [ 3.47704179e+03, -1.44896703e+03,  9.87875540e+02]]],


        [[[-3.86844741e+02,  2.29019708e+03,  1.96014373e+02],
          [-7.30902615e+02,  4.95870424e+03, -4.43772741e+03],
          [ 2.20599508e+03,  5.51802422e+02,  4.86477148e+02]],

         [[ 2.23177550e+03,  3.14088224e+02, -1.63369238e+03],
          [ 1.11700217e+03,  4.16006680e+03,  4.32240144e+02],
          [ 3.01297779e+03, -2.72446934e+03, -2.53466192e+03]]],


        [[[-2.88254629e+03,  5.63274258e+03, -3.57488136e+03],
          [ 7.69629778e+02, -3.93041066e+01, -1.39262850e+03],
...
          [ 7.91131626e+03, -6.93166562e+03,  9.92489146e+03]]],


        [[[ 1.54797401e+04,  2.10456671e+04, -1.29688305e+01],
          [-1.65451408e+04, -2.00068142e+04, -1.39894224e+03],
          [-1.32234749e+02,  4.24279563e+03, -4.31500262e+03]],

         [[ 1.35742501e+04, -1.69390891e+04,  3.20391137e+02],
          [-2.81371399e+04,  8.06252184e+03,  5.80997510e+03],
          [ 1.10832097e+04, -9.62765048e+03,  1.18229510e+04]]],


        [[[-3.78914818e+03,  9.54658467e+04,  4.41247122e+03],
          [-1.11313031e+05, -1.07830702e+05,  6.72671093e+04],
          [ 1.58173559e+05, -4.12625195e+04, -1.18398451e+05]],

         [[-2.65912793e+05, -1.30407411e+05, -8.59456652e+03],
          [ 4.89005957e+05,  2.06822260e+05,  8.71720662e+03],
          [-5.46576457e+05, -2.80688618e+05,  2.66678802e+05]]]]],
      shape=(378, 3, 2, 3, 3))
Coordinates:
  * gridcell          (gridcell) int64 3kB 0 1 2 3 4 5 ... 373 374 375 376 377
  * depth             (depth) float64 24B 0.03 0.19 0.78
  * hist              (hist) int64 16B 0 1
  * lon_translations  (lon_translations) int64 24B -1 0 1
  * lat_translations  (lat_translations) int64 24B -1 0 1

In [23]:
var_predictor_sets["radius_1"].pr 

<xarray.DataArray 'pr' (lat_translations: 3, lon_translations: 3, time: 165,
                        hist: 2, gridcell: 378)> Size: 9MB
array([[[[[9.79363087e-06, 6.84418668e-06, 1.69429628e-05, ...,
           3.91690886e-06, 3.24777212e-06, 2.93179228e-06],
          [9.48900112e-06, 5.03962735e-06, 1.78231419e-06, ...,
           3.50556166e-06, 3.01161972e-06, 2.45377107e-06]],

         [[9.60281598e-06, 5.34766110e-06, 2.03984659e-06, ...,
           1.65236675e-06, 1.53216917e-06, 2.25439313e-06],
          [1.38725365e-05, 2.58153608e-05, 3.58273938e-05, ...,
           3.72799884e-06, 2.69548961e-06, 2.57786600e-06]],

         [[1.62397223e-05, 1.32865902e-05, 1.30303422e-05, ...,
           2.62350909e-06, 2.40706393e-06, 2.54728931e-06],
          [1.23923234e-05, 2.35007885e-05, 2.66705030e-05, ...,
           4.85800771e-06, 4.84055039e-06, 3.61957662e-06]],

         ...,

         [[2.00124051e-05, 4.30717812e-06, 3.37423279e-06, ...,
           4.85082381e-06, 3.97809164e-06, 4.89425248e-06],
          [9.66114042e-06, 4.60282600e-06, 4.44213349e-06, ...,
...
          [3.36036455e-05, 3.62344605e-05, 3.76311707e-05, ...,
           3.66242375e-06, 5.87960304e-06, 4.54475719e-06]],

         ...,

         [[2.88067923e-05, 4.67860629e-05, 4.92371723e-05, ...,
           3.33519252e-06, 3.95666371e-06, 1.90449003e-06],
          [3.62071466e-05, 4.33659020e-05, 4.08792273e-05, ...,
           3.88590987e-06, 4.55633971e-06, 7.42229868e-06]],

         [[2.97530410e-05, 4.80534804e-05, 4.21707081e-05, ...,
           4.08633953e-06, 5.58263845e-06, 2.86737683e-06],
          [2.64298738e-05, 4.29243390e-05, 3.84903609e-05, ...,
           2.33693715e-06, 3.30332502e-06, 2.91549326e-06]],

         [[3.96400147e-05, 4.11227465e-05, 3.69766894e-05, ...,
           1.69997923e-06, 2.55058299e-06, 3.69480352e-06],
          [4.35822750e-05, 4.09462767e-05, 4.17459049e-05, ...,
           5.58235956e-06, 5.10038694e-06, 3.69006022e-06]]]]],
      shape=(3, 3, 165, 2, 378))
Coordinates:
  * lat_translations  (lat_translations) int64 24B -1 0 1
  * lon_translations  (lon_translations) int64 24B -1 0 1
  * time              (time) datetime64[ns] 1kB 1850-04-30 ... 2014-04-30
  * hist              (hist) int64 16B 0 1
    lat               (gridcell) float64 3kB -51.75 -47.25 ... 78.75 78.75
    lon               (gridcell) float64 3kB 288.0 288.0 288.0 ... 279.0 297.0
    height            float64 8B 2.0
Dimensions without coordinates: gridcell
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [24]:
Regr_set_mean["radius_2"].params.pr * var_predictor_sets["radius_2"].pr 

<xarray.DataArray 'pr' (gridcell: 378, depth: 3, hist: 2, lon_translations: 5,
                        lat_translations: 5, time: 165)> Size: 75MB
array([[[[[[ 6.51091304e-02,  4.56485899e-02,  6.81933193e-02, ...,
             1.05063229e-01,  1.48333799e-01,  7.00415718e-02],
           [-2.27306155e-02, -2.35378076e-02, -2.84581022e-02, ...,
            -3.88875755e-02, -3.60800740e-02, -3.58935827e-02],
           [ 3.12792771e-02,  4.94578063e-02,  3.79875932e-02, ...,
             6.18510451e-02,  4.03743718e-02,  5.03978322e-02],
           [-7.30841078e-02, -8.69158440e-02, -6.02616453e-02, ...,
            -8.40865551e-02, -4.87058761e-02, -6.77431548e-02],
           [ 6.52049307e-05,  8.38802920e-05,  4.74750241e-05, ...,
             3.86756247e-05,  4.76865098e-05,  5.24212149e-05]],

          [[-3.42041394e-03, -2.67251836e-03, -6.64003487e-03, ...,
            -2.15253218e-03, -1.11718365e-02, -4.84776893e-03],
           [-1.19164595e-02, -1.16842844e-02, -1.97597803e-02, ...,
            -2.43502148e-02, -3.95381964e-02, -1.40227046e-02],
           [ 2.88770524e-02,  3.95251938e-02,  4.79176254e-02, ...,
             7.38675414e-02,  7.71990913e-02,  4.90325841e-02],
           [ 1.16951254e-01,  1.31854316e-01,  9.89344145e-02, ...,
             1.21097398e-01,  1.07015641e-01,  1.01944343e-01],
           [-4.67412711e-02, -4.77789107e-02, -4.27739632e-02, ...,
...
           [-2.67465401e+00, -2.39389524e+00, -4.29894832e+00, ...,
            -2.27101739e+00, -5.88516320e+00, -6.13694427e+00],
           [-4.42353989e+00, -2.19498864e+00, -5.81799159e+00, ...,
            -3.15852621e+00, -1.74122056e+00, -4.99846092e+00],
           [-6.92449163e-01, -3.24209956e-01, -4.96699483e-01, ...,
            -8.11187873e-01, -3.18636164e-01, -4.03289093e-01],
           [ 1.72557878e+00,  9.88093655e-01,  8.50970805e-01, ...,
             7.61231956e-01,  1.05982306e+00,  6.88195843e-01]],

          [[ 3.81273685e+00,  3.86775847e+00,  3.14336099e+00, ...,
             6.00828717e+00,  3.52937810e+00,  8.13908828e+00],
           [ 1.03575230e+00,  1.10147353e+00,  1.43534564e+00, ...,
             9.52136923e-01,  1.97369073e+00,  2.49727709e+00],
           [ 1.03459386e+00,  4.90975761e-01,  1.16225118e+00, ...,
             1.23455583e+00,  1.87309810e-01,  1.01744889e+00],
           [ 4.21243774e-01,  2.11672566e-01,  3.77232435e-01, ...,
             4.35917885e-01,  2.79553045e-01,  2.46123335e-01],
           [ 6.74106139e-01,  3.33774954e-01,  3.51777962e-01, ...,
             5.52967834e-01,  4.86038765e-01,  4.08056084e-01]]]]]],
      shape=(378, 3, 2, 5, 5, 165))
Coordinates:
  * gridcell          (gridcell) int64 3kB 0 1 2 3 4 5 ... 373 374 375 376 377
    lat               (gridcell) float64 3kB -51.75 -47.25 ... 78.75 78.75
    lon               (gridcell) float64 3kB 288.0 288.0 288.0 ... 279.0 297.0
  * depth             (depth) float64 24B 0.03 0.19 0.78
  * hist              (hist) int64 16B 0 1
  * lon_translations  (lon_translations) int64 40B -2 -1 0 1 2
  * lat_translations  (lat_translations) int64 40B -2 -1 0 1 2
  * time              (time) datetime64[ns] 1kB 1850-04-30 ... 2014-04-30
    height            float64 8B 2.0
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [25]:
residuals = {}
for key, regr in Regr_set_mean.items():
    residuals[key] = regr.residuals(var_predictor_sets[key], var_target,location_dim="gridcell", regr_dim="time")


### Linear Regression of the Variance

In [26]:
Regr_set_var = {}

In [27]:
for key, res in residuals.items():
    Regr_set_var[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist","lon_translations","lat_translations"])
    Regr_set_var[key].fit(predictors=var_predictor_sets[key], target=(res.residuals)**2,location_dim="gridcell", regr_dim="time")

for simplicity not in use
### Compute skewness samples
skew_target_noneT = shape_target(raw_mrsol_for_skew, approx_maximas)
skew_target_noneT, chunk_mask_skew, detail_mask_skew = mask_stack_target(skew_target_noneT)
skew_target = model.transform.Logit_Transform_ds(skew_target_noneT)

skew_predictors = shape_input(raw_input_for_skew,chunk_mask)
mean_prediction = LinReg_mean.predict(skew_predictors)
residuals = skew_target.mrsol - mean_prediction.prediction
sigmas = np.sqrt(LinReg_variance.predict(skew_predictors).prediction.clip(min = 1e-32))
standardized_values_for_skew = (residuals/sigmas)
### Linear Regression of the Skewness
LinReg_skewness = model.stats._parallel_linear_regression.ParLinearRegression()
LinReg_skewness.fit(predictors=skew_predictors, target=(standardized_values_for_skew)**3,location_dim="gridcell", regr_dim="time")

### Predictor

### Export Prameters

In [28]:
if safe:
    for key, mean_regr in Regr_set_mean.items():
        model.save.save_params(mean_regr.params,Regr_set_var[key].params, maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/radius/{key}/local{is_local_data}/month{Month_idx}", name=f"start={start_wanted},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)
